<a href="https://colab.research.google.com/github/ompatel420/FreeCodeCamp-Machine-Learning-Project/blob/main/Rock_Paper_Scissors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/freeCodeCamp/boilerplate-rock-paper-scissors.git


Cloning into 'boilerplate-rock-paper-scissors'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 25 (delta 6), reused 2 (delta 2), pack-reused 9 (from 2)
Receiving objects: 100% (25/25), 6.89 KiB | 1.15 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
%cd /content/boilerplate-rock-paper-scissors

/content/boilerplate-rock-paper-scissors


In [3]:
!ls

main.py  README.md  RPS_game.py  RPS.py  test_module.py


In [4]:
%%writefile RPS.py

import random
from collections import defaultdict, Counter

def player(prev_play, opponent_history=[]):

    # Reset the data at the beginning of a new match
    if prev_play == "":
        opponent_history.clear()
        player.my_history = []
        player.transitions = defaultdict(Counter)
        player.context_counts = defaultdict(Counter)

    # Initialize storage
    if not hasattr(player, "my_history"):
        player.my_history = []
        player.transitions = defaultdict(Counter)
        player.context_counts = defaultdict(Counter)

    # Store opponent's previous move
    if prev_play:
        opponent_history.append(prev_play)

    # Store the previous move pair:
    # Our previous move -> opponent's previous response
    if prev_play and player.my_history:
        last_our_move = player.my_history[-1]

        player.transitions[last_our_move][prev_play] += 1

        # Learn responses to the last two moves we played
        if len(player.my_history) >= 2:
            context = "".join(player.my_history[-2:])
            player.context_counts[context][prev_play] += 1

    moves = ["R", "P", "S"]

    # Move that defeats each possible opponent move
    counter = {
        "R": "P",
        "P": "S",
        "S": "R"
    }

    # Predict the opponent's next move
    predictions = Counter()

    # Strategy 1: Predict repeated opponent patterns
    history = opponent_history

    for order in range(1, min(5, len(history)) + 1):

        pattern = tuple(history[-order:])

        for i in range(len(history) - order):

            if tuple(history[i:i + order]) == pattern:

                if i + order < len(history):
                    next_move = history[i + order]

                    weight = order ** 2
                    predictions[next_move] += weight

    # Strategy 2: Learn opponent response to our previous move
    if player.my_history:

        last_our_move = player.my_history[-1]

        response_counts = player.transitions[last_our_move]

        for move, count in response_counts.items():
            predictions[move] += count * 2

    # Strategy 3: Learn opponent response to our last two moves
    if len(player.my_history) >= 2:

        context = "".join(player.my_history[-2:])

        response_counts = player.context_counts[context]

        for move, count in response_counts.items():
            predictions[move] += count * 4

    # Strategy 4: Detect opponent's most frequent move
    if len(opponent_history) >= 10:

        recent_moves = opponent_history[-10:]
        most_common = Counter(recent_moves).most_common(1)[0][0]

        predictions[most_common] += 1

    # Select a predicted move
    if predictions:

        predicted_move = predictions.most_common(1)[0][0]

        # Play the move that defeats the prediction
        next_move = counter[predicted_move]

    else:

        # Random move when there is insufficient information
        next_move = random.choice(moves)

    # Store our move for future learning
    player.my_history.append(next_move)

    return next_move

Overwriting RPS.py


In [5]:
%%writefile main.py

from RPS_game import play, quincy, abbey, kris, mrugesh
from RPS import player

print("\n========== ROCK PAPER SCISSORS AI ==========\n")

print("\nTesting against Quincy:")
play(player, quincy, 1000)

print("\nTesting against Abbey:")
play(player, abbey, 1000)

print("\nTesting against Kris:")
play(player, kris, 1000)

print("\nTesting against Mrugesh:")
play(player, mrugesh, 1000)

print("\n========== TESTING COMPLETED ==========")

Overwriting main.py


In [6]:
!python main.py


========== ROCK PAPER SCISSORS AI ==========


Testing against Quincy:
Final results: {'p1': 996, 'p2': 0, 'tie': 4}
Player 1 win rate: 100.0%

Testing against Abbey:
Final results: {'p1': 437, 'p2': 278, 'tie': 285}
Player 1 win rate: 61.11888111888112%

Testing against Kris:
Final results: {'p1': 774, 'p2': 181, 'tie': 45}
Player 1 win rate: 81.04712041884817%

Testing against Mrugesh:
Final results: {'p1': 844, 'p2': 155, 'tie': 1}
Player 1 win rate: 84.48448448448448%

========== TESTING COMPLETED ==========


In [7]:
!python -m unittest test_module -v


test_player_vs_abbey (test_module.UnitTests.test_player_vs_abbey) ... Testing game against abbey...
Final results: {'p1': 437, 'p2': 278, 'tie': 285}
Player 1 win rate: 61.11888111888112%
ok
test_player_vs_kris (test_module.UnitTests.test_player_vs_kris) ... Testing game against kris...
Final results: {'p1': 680, 'p2': 250, 'tie': 70}
Player 1 win rate: 73.11827956989248%
ok
test_player_vs_mrugesh (test_module.UnitTests.test_player_vs_mrugesh) ... Testing game against mrugesh...
Final results: {'p1': 836, 'p2': 164, 'tie': 0}
Player 1 win rate: 83.6%
ok
test_player_vs_quincy (test_module.UnitTests.test_player_vs_quincy) ... Testing game against quincy...
Final results: {'p1': 996, 'p2': 0, 'tie': 4}
Player 1 win rate: 100.0%
ok

----------------------------------------------------------------------
Ran 4 tests in 2.642s

OK


In [8]:
%%writefile RPS.py

import random
from collections import defaultdict, Counter

def player(prev_play, opponent_history=[]):

    if prev_play == "":
        opponent_history.clear()
        player.my_history = []
        player.transitions = defaultdict(Counter)
        player.context_counts = defaultdict(Counter)

    if not hasattr(player, "my_history"):
        player.my_history = []
        player.transitions = defaultdict(Counter)
        player.context_counts = defaultdict(Counter)

    if prev_play:
        opponent_history.append(prev_play)

    # Learn from the opponent's previous response.
    # The game engine introduces a one-round delay.
    if prev_play and len(player.my_history) >= 2:

        previous_our_move = player.my_history[-2]

        player.transitions[previous_our_move][prev_play] += 1

        if len(player.my_history) >= 3:
            context = "".join(player.my_history[-3:-1])
            player.context_counts[context][prev_play] += 1

    counter = {
        "R": "P",
        "P": "S",
        "S": "R"
    }

    predictions = Counter()
    history = opponent_history

    # 1. N-gram pattern recognition
    for order in range(1, min(5, len(history)) + 1):

        pattern = tuple(history[-order:])

        for i in range(len(history) - order):

            if tuple(history[i:i + order]) == pattern:

                if i + order < len(history):

                    next_move = history[i + order]
                    predictions[next_move] += order ** 2

    # 2. Predict opponent response to our previous move
    if player.my_history:

        last_move = player.my_history[-1]

        for move, count in player.transitions[last_move].items():
            predictions[move] += count * 2

    # 3. Predict using our last two moves
    if len(player.my_history) >= 2:

        context = "".join(player.my_history[-2:])

        for move, count in player.context_counts[context].items():
            predictions[move] += count * 4

    # 4. Predict Mrugesh's frequency-based response
    if player.my_history:

        recent = player.my_history[-10:]

        most_common = Counter(recent).most_common(1)[0][0]

        predictions[counter[most_common]] += 3

    # Select the predicted opponent move
    if predictions:

        predicted_move = predictions.most_common(1)[0][0]
        next_move = counter[predicted_move]

    else:

        next_move = random.choice(["R", "P", "S"])

    player.my_history.append(next_move)

    return next_move

Overwriting RPS.py


In [9]:
!python main.py


========== ROCK PAPER SCISSORS AI ==========


Testing against Quincy:
Final results: {'p1': 996, 'p2': 1, 'tie': 3}
Player 1 win rate: 99.89969909729187%

Testing against Abbey:
Final results: {'p1': 554, 'p2': 295, 'tie': 151}
Player 1 win rate: 65.2532391048292%

Testing against Kris:
Final results: {'p1': 991, 'p2': 9, 'tie': 0}
Player 1 win rate: 99.1%

Testing against Mrugesh:
Final results: {'p1': 839, 'p2': 161, 'tie': 0}
Player 1 win rate: 83.89999999999999%

========== TESTING COMPLETED ==========


In [10]:
!python -m unittest test_module -v


test_player_vs_abbey (test_module.UnitTests.test_player_vs_abbey) ... Testing game against abbey...
Final results: {'p1': 641, 'p2': 344, 'tie': 15}
Player 1 win rate: 65.0761421319797%
ok
test_player_vs_kris (test_module.UnitTests.test_player_vs_kris) ... Testing game against kris...
Final results: {'p1': 991, 'p2': 9, 'tie': 0}
Player 1 win rate: 99.1%
ok
test_player_vs_mrugesh (test_module.UnitTests.test_player_vs_mrugesh) ... Testing game against mrugesh...
Final results: {'p1': 837, 'p2': 160, 'tie': 3}
Player 1 win rate: 83.95185556670009%
ok
test_player_vs_quincy (test_module.UnitTests.test_player_vs_quincy) ... Testing game against quincy...
Final results: {'p1': 996, 'p2': 1, 'tie': 3}
Player 1 win rate: 99.89969909729187%
ok

----------------------------------------------------------------------
Ran 4 tests in 2.731s

OK


In [12]:
import random

def play_game():

    choices = {
        "R": "Rock",
        "P": "Paper",
        "S": "Scissors"
    }

    print("===== ROCK PAPER SCISSORS GAME =====")

    while True:

        user = input("\nEnter R for Rock, P for Paper, S for Scissors (Q to quit): ").upper()

        if user == "Q":
            print("Thanks for playing!")
            break

        if user not in choices:
            print("Invalid choice! Try again.")
            continue

        computer = random.choice(["R", "P", "S"])

        print("You selected:", choices[user])
        print("Computer selected:", choices[computer])

        if user == computer:
            print("Result: It's a Tie!")

        elif (user == "R" and computer == "S") or \
             (user == "P" and computer == "R") or \
             (user == "S" and computer == "P"):

            print("Result: You Win!")

        else:
            print("Result: Computer Wins!")

play_game()

===== ROCK PAPER SCISSORS GAME =====

Enter R for Rock, P for Paper, S for Scissors (Q to quit): a
Invalid choice! Try again.

Enter R for Rock, P for Paper, S for Scissors (Q to quit): s
You selected: Scissors
Computer selected: Scissors
Result: It's a Tie!

Enter R for Rock, P for Paper, S for Scissors (Q to quit): r
You selected: Rock
Computer selected: Scissors
Result: You Win!

Enter R for Rock, P for Paper, S for Scissors (Q to quit): p
You selected: Paper
Computer selected: Rock
Result: You Win!

Enter R for Rock, P for Paper, S for Scissors (Q to quit): q
Thanks for playing!
